# DreamerV4 Code Walkthrough: Training Agents Inside Scalable World Models

This notebook provides a deep dive into the core components of the DreamerV4 PyTorch implementation. Instead of standard recurrent models (like the RSSMs in DreamerV1-V3), DreamerV4 achieves state-of-the-art sample efficiency and scalability using **Block-Causal Transformers** and **Shortcut Forcing**.

We will explore how pixels become patches, how patches become latents, how the agent imagines the future, and how those imaginations are decoded back into pixels.

## 1. Setup & Initialization
Let's import PyTorch and the network modules. We'll set up a dummy configuration that matches the shapes used during inference/training.

In [3]:
import sys
import os
# Add the inner dreamer4 directory to path so absolute imports inside the repo (like 'from model import ...') work
sys.path.append(os.path.abspath('./dreamer4/dreamer4'))

import torch
from model import (
    Encoder, Decoder, Dynamics, 
    temporal_patchify, temporal_unpatchify, BlockCausalTransformer
)
from interactive import make_tau_schedule, sample_one_timestep_packed

# Configuration (Lightweight dummy sizes for demonstration)
B = 2               # Batch size
T = 4               # Sequence length (Time)
C, H, W = 3, 64, 64 # Image shape (RGB, 64x64)
patch = 8           # Patch size (8x8 pixels per patch)
n_patches = (H // patch) * (W // patch) # Total patches per frame: 64
d_patch = patch * patch * C             # Dimensionality of one patch: 192

d_model = 128       # Transformer hidden dimension
n_latents = 8       # Number of discrete latents to compress the image into
d_bottleneck = 16   # Dimensionality of the bottleneck
packing_factor = 2  # Used for packing latents efficiently in the Dynamics model

print(f"Image Shape: {C}x{H}x{W}")
print(f"Number of Patches per frame: {n_patches}")
print(f"Latent dimensions (d_model): {d_model}")

Image Shape: 3x64x64
Number of Patches per frame: 64
Latent dimensions (d_model): 128


## 2. Temporal Patchification
Transformers don't work efficiently on raw pixels. Similar to Vision Transformers (ViT), DreamerV4 splits images into grid patches. `temporal_patchify` takes a sequence of images `(B, T, C, H, W)` and reshapes them into `(B, T, NumPatches, PatchDim)`.

In [4]:
# Create a dummy sequence of frames (e.g. from the environment)
frames = torch.rand(B, T, C, H, W)
print(f"Original frames shape: {frames.shape}")

# Patchify the frames
patches = temporal_patchify(frames, patch)
print(f"Patchified shape: {patches.shape}")
print(f"-> Meaning: {B} batches, {T} timesteps, {n_patches} patches per frame, {d_patch} features per patch")

Original frames shape: torch.Size([2, 4, 3, 64, 64])
Patchified shape: torch.Size([2, 4, 64, 192])
-> Meaning: 2 batches, 4 timesteps, 64 patches per frame, 192 features per patch


## 3. The Encoder & Masked Auto-Encoding (MAE)
The **Encoder** takes these patches and compresses them into `n_latents`. 

**Crucial Concept:** To ensure the encoder learns deep semantic features instead of just memorizing pixels, it uses a `MAEReplacer`. During training, up to 90% of the patches are replaced by a learnable `[MASK]` token. The model is forced to reconstruct the missing pieces, forcing the latent `z` to truly understand the scene.

In [6]:
# Initialize the Encoder with MAE masking enabled (mae_p_max=0.9)
encoder = Encoder(
    patch_dim=d_patch, 
    d_model=d_model, 
    n_latents=n_latents, 
    n_patches=n_patches,
    n_heads=4, depth=2, 
    d_bottleneck=d_bottleneck, 
    time_every=1,
    mae_p_min=0.5, 
    mae_p_max=0.9 # Between 50% and 90% of patches are masked!
)

# Run the patches through the encoder
z_latents, (mae_mask, keep_prob) = encoder(patches)

print(f"Encoded Latent 'z' Shape: {z_latents.shape}")
print(f"Notice that 'z' has been compressed from {n_patches} patches down to just {n_latents} latents!")
print(f"\nMAE Mask Shape: {mae_mask.shape} (True where patches were hidden)")

Encoded Latent 'z' Shape: torch.Size([2, 4, 8, 16])
Notice that 'z' has been compressed from 64 patches down to just 8 latents!

MAE Mask Shape: torch.Size([2, 4, 64, 1]) (True where patches were hidden)


## 4. The Block-Causal Transformer
At the heart of the Encoder, Decoder, and Dynamics models is the `BlockCausalTransformer`. This replaces the recurrent RSSM used in previous Dreamer versions.

It alternates between:
1. **SpaceSelfAttention**: Mixes information across modalities (latents, images, actions) *within the same timestep*.
2. **TimeSelfAttention**: Mixes information *across timesteps*, ensuring causal behavior (you can't look into the future).

In [7]:
print(encoder.transformer)
# Notice in the output below how it stacks BlockCausalLayers.
# Each layer has a SpaceSelfAttentionModality, followed periodically by a TimeSelfAttention.

BlockCausalTransformer(
  (layers): ModuleList(
    (0-1): 2 x BlockCausalLayer(
      (norm1): RMSNorm()
      (space): SpaceSelfAttentionModality(
        (attn): MultiheadSelfAttention(
          (qkv): Linear(in_features=128, out_features=384, bias=True)
          (out): Linear(in_features=128, out_features=128, bias=True)
        )
      )
      (drop1): Dropout(p=0.0, inplace=False)
      (norm2): RMSNorm()
      (time): TimeSelfAttention(
        (attn): MultiheadSelfAttention(
          (qkv): Linear(in_features=128, out_features=384, bias=True)
          (out): Linear(in_features=128, out_features=128, bias=True)
        )
      )
      (drop2): Dropout(p=0.0, inplace=False)
      (norm3): RMSNorm()
      (mlp): MLP(
        (fc_in): Linear(in_features=128, out_features=1024, bias=True)
        (fc_out): Linear(in_features=512, out_features=128, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
)


## 5. The Dynamics Model (The World Simulator)
The Dynamics model predicts the *next* latent state given past states and continuous actions.

Because DreamerV4 scales to huge models, autoregressively predicting frame-by-frame is slow. They introduce **Shortcut Forcing**: a denoising schedule (`tau`) where the model predicts jumps into the future.

In [8]:
# Initialize Dynamics
dynamics = Dynamics(
    d_model=d_model, 
    d_bottleneck=d_bottleneck, 
    d_spatial=d_bottleneck * packing_factor, n_spatial=n_latents // packing_factor, 
    n_register=4,
    n_agent=0,
    n_heads=4, 
    depth=2, 
    k_max=8
)

# Pack the latents for efficiency
n_spatial = n_latents // packing_factor
d_spatial = d_bottleneck * packing_factor
packed_z = z_latents.view(B, T, n_spatial, packing_factor, d_bottleneck).reshape(B, T, n_spatial, d_spatial)
print(f"Packed Spatial State: {packed_z.shape}")

# Dummy continuous actions (e.g. 16-DOF robot joints in [-1, 1])
actions = torch.rand(B, T, 16) * 2 - 1

# The schedule tells the model 'how far' into the future to imagine
schedule = make_tau_schedule(k_max=8, schedule="finest")
step_idxs = torch.ones(B, T, dtype=torch.long) * schedule['e']
signal_idxs = torch.ones(B, T, dtype=torch.long) * 7

predicted_x1, _ = dynamics(actions, step_idxs, signal_idxs, packed_z)
print(f"Imagined Future Latent State: {predicted_x1.shape}")

Packed Spatial State: torch.Size([2, 4, 4, 32])
Imagined Future Latent State: torch.Size([2, 4, 4, 32])


## 6. The Decoder (Bringing Imagination to Reality)
Finally, to visualize what the agent is imagining (or to compute the reconstruction loss during training), we pass the predicted latents through the `Decoder` and `temporal_unpatchify` to get back an RGB image.

In [10]:
decoder = Decoder(
    d_bottleneck=d_bottleneck, 
    d_model=d_model, 
    n_heads=4, 
    depth=2,
    n_latents=n_latents, 
    n_patches=n_patches, 
    d_patch=d_patch, 
    time_every=1
)

# Decode the latents back into patches
reconstructed_patches = decoder(z_latents)
print(f"Reconstructed Patches: {reconstructed_patches.shape}")

# Fold the patches back into standard Images (C, H, W)
imagined_images = temporal_unpatchify(reconstructed_patches, H, W, C, patch)
print(f"Final Output Image: {imagined_images.shape}")
print("Success! Mapped raw pixels -> latent compression -> dynamics prediction -> pixel reconstruction.")

Reconstructed Patches: torch.Size([2, 4, 64, 192])
Final Output Image: torch.Size([2, 4, 3, 64, 64])
Success! Mapped raw pixels -> latent compression -> dynamics prediction -> pixel reconstruction.
